In [1]:
# ============================================================
# Dataset: Symptom2Disease (Colab)
#
# Objectif :
# - Classification sur 14 maladies
# - Même pipeline / mêmes métriques pour toutes les variantes
# - P0 / P1 / P2 (sans CoT)
# - P3.0 / P3.1 / P3.2 / P3.3 (CoT ablation)
# ============================================================

import re, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

/Users/camilleb/Desktop/ING3/MLOps/NLP-classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# 0) Reproductibilité
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# ----------------------------
# 1) Configuration
# ----------------------------
STOD_PATH = "./content/Symptom2Disease.csv"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# On garde les "top maladies" (fréquences) puis on équilibre à MAX_PER_DISEASE par maladie.
TOP_K_DISEASES = 15
MAX_PER_DISEASE = 50

TEST_SIZE = 0.2
VAL_SIZE = 0.2  # part du TRAIN utilisée comme validation (train -> train/val)

# Few-shot dynamique (P2 / P3.3)
DYN_K = 4
DYN_PER_CLASS_CAP = 1

In [4]:
# ----------------------------
# 2) Chargement du dataset
# ----------------------------
df_all = pd.read_csv(STOD_PATH)
if "Unnamed: 0" in df_all.columns:
    df_all = df_all.drop(columns=["Unnamed: 0"])

df_all = df_all.rename(columns={"label": "disease", "text": "symptom_text"})
df_all["disease"] = df_all["disease"].astype(str)
df_all["symptom_text"] = df_all["symptom_text"].astype(str)

In [5]:
# ----------------------------
# 3) Sélection des TOP_K_DISEASES + équilibrage
# ----------------------------
topk = df_all["disease"].value_counts().index[:TOP_K_DISEASES].tolist()
df_sel = df_all[df_all["disease"].isin(topk)].copy()

# Équilibrage : on limite à MAX_PER_DISEASE exemples par maladie (pas de groupby.apply)
parts = []
for disease, g in df_sel.groupby("disease"):
    parts.append(g.sample(min(len(g), MAX_PER_DISEASE), random_state=SEED))
df = pd.concat(parts, ignore_index=True)

print("Rows kept:", len(df))
print(df["disease"].value_counts())

Rows kept: 750
disease
Acne                     50
Arthritis                50
Bronchial Asthma         50
Chicken pox              50
Common Cold              50
Dengue                   50
Dimorphic Hemorrhoids    50
Fungal infection         50
Hypertension             50
Impetigo                 50
Migraine                 50
Pneumonia                50
Psoriasis                50
Typhoid                  50
Varicose Veins           50
Name: count, dtype: int64


In [6]:
# ----------------------------
# 4) Split Train / Test puis Train -> Train/Val
# ----------------------------
X = df[["symptom_text"]]
y = df["disease"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train_full
)

print("\nSplit sizes:")
print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))


Split sizes:
Train: 480 Val: 120 Test: 150


In [7]:
# ----------------------------
# 5) Encodage des classes : L00..L14 (codes)
# ----------------------------
labels = sorted(df["disease"].unique().tolist())
N_CLASSES = len(labels)

def code(i: int) -> str:
    return f"L{i:02d}"

idx2label = {i: labels[i] for i in range(N_CLASSES)}
label2idx = {labels[i]: i for i in range(N_CLASSES)}
idx2code  = {i: code(i) for i in range(N_CLASSES)}
code2idx  = {code(i): i for i in range(N_CLASSES)}

# Bloc d'options affiché dans les prompts
OPTIONS_BLOCK = "\n".join([f"{idx2code[i]}) {idx2label[i]}" for i in range(N_CLASSES)])
print("\nDISEASE CLASSES:\n", OPTIONS_BLOCK)


DISEASE CLASSES:
 L00) Acne
L01) Arthritis
L02) Bronchial Asthma
L03) Chicken pox
L04) Common Cold
L05) Dengue
L06) Dimorphic Hemorrhoids
L07) Fungal infection
L08) Hypertension
L09) Impetigo
L10) Migraine
L11) Pneumonia
L12) Psoriasis
L13) Typhoid
L14) Varicose Veins


In [8]:
# ----------------------------
# 6) Chargement du modèle
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)
model.eval()

def chat_template(system_msg: str, user_msg: str) -> str:
    """
    Applique le "chat template" du modèle (format instruct/chat).
    """
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_text(prompt_text: str, max_new_tokens: int = 50) -> str:
    """
    Génère la sortie du modèle (greedy, sans sampling).
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:11<00:00, 36.87it/s]
Some parameters are on the meta device because they were offloaded to the disk.


In [9]:
# ----------------------------
# 7) Few-shot statique : 1 exemple par maladie (pour P3.2)
# ----------------------------
def build_fewshot_one_per_class(Xtr_df: pd.DataFrame, ytr_series: pd.Series, seed: int = 42):
    """
    Construit une liste d'exemples: (texte, code) avec exactement 1 exemple par maladie.
    Utilisé pour la variante P3.2 (CoT + few-shot statique).
    """
    rng = np.random.RandomState(seed)
    out = []
    for lab in labels:
        subset = Xtr_df[ytr_series == lab]
        row = subset.sample(1, random_state=int(rng.randint(0, 10**9))).iloc[0]
        txt = row["symptom_text"].strip().replace("\n", " ")
        out.append((txt, idx2code[label2idx[lab]]))
    return out

FEWSHOT_STATIC = build_fewshot_one_per_class(X_train, y_train, seed=SEED)
FEWSHOT_STATIC_TEXT = ""
for txt, c in FEWSHOT_STATIC:
    FEWSHOT_STATIC_TEXT += f"SYMPTOM: {txt}\nLABEL: {c}\n\n"

In [10]:
# ----------------------------
# 8) Few-shot dynamique TF-IDF (pour P2 & P3.3)
# ----------------------------
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95)
Xtr_mat = tfidf.fit_transform(X_train["symptom_text"].tolist())

train_texts = X_train["symptom_text"].tolist()
train_labels = y_train.tolist()

def fewshot_dynamic(txt: str, k: int = DYN_K, per_class_cap: int = DYN_PER_CLASS_CAP) -> str:
    """
    Va chercher des exemples "proches" (similarité TF-IDF) dans le train
    et les insère dans le prompt.

    - k : nombre total d'exemples retournés
    - per_class_cap : limite d'exemples par maladie, pour éviter d'avoir 4 fois la même maladie
    """
    q = tfidf.transform([txt])
    sims = cosine_similarity(q, Xtr_mat)[0]
    idxs = np.argsort(-sims)

    chosen, counts = [], {}
    for i in idxs:
        lab = train_labels[i]
        if counts.get(lab, 0) >= per_class_cap:
            continue
        chosen.append((train_texts[i], idx2code[label2idx[lab]]))
        counts[lab] = counts.get(lab, 0) + 1
        if len(chosen) >= k:
            break

    block = ""
    for t, c in chosen:
        block += f"SYMPTOM: {t}\nLABEL: {c}\n\n"
    return block

In [11]:
# ----------------------------
# 9) Parsing robuste : "LABEL: Lxx" OU "Lxx"
#     On prend le DERNIER match (le modèle peut écrire plusieurs labels)
# ----------------------------
CODE_RE = re.compile(r"(?:LABEL\s*[:=\-]\s*)?(L\d{2})\b", re.IGNORECASE)

def extract_code_last(text: str):
    matches = CODE_RE.findall(text)
    if not matches:
        return None
    return matches[-1].upper()

In [12]:
# ----------------------------
# 10) Fonction de score (mêmes métriques partout)
# ----------------------------
def score_metrics(y_true, y_pred):
    """
    Calcule des métriques identiques pour toutes les variantes de prompt.

    1) accuracy
       - Proportion de prédictions correctes (0.0 à 1.0)
       - Exemple: 0.53 = 53% des textes correctement classés

    2) macro precision / recall / f1
       - On calcule P/R/F1 pour CHAQUE maladie, puis on fait une moyenne simple.
       - Chaque maladie compte autant (même si elle est rare/fréquente).
       - Très utile pour détecter si le modèle est bon "globalement" sur toutes les maladies.

    3) weighted precision / recall / f1
       - Même chose, mais la moyenne est pondérée par le "support" (nombre d'exemples par classe).
       - Si certaines maladies sont plus fréquentes, elles comptent plus.

    Note:
    - zero_division=0 évite des warnings quand le modèle ne prédit jamais une classe.
    """
    acc = accuracy_score(y_true, y_pred)

    pM, rM, fM, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    pW, rW, fW, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "accuracy": float(acc),
        "precision_macro": float(pM),
        "recall_macro": float(rM),
        "f1_macro": float(fM),
        "precision_weighted": float(pW),
        "recall_weighted": float(rW),
        "f1_weighted": float(fW),
    }

def print_eval_block(name: str, y_true, y_pred, fails: int = 0, show_report: bool = True):
    """
    Affiche un bloc de résultats uniforme pour comparer les variantes.

    - fails = nombre de fois où on n'a pas réussi à parser un code Lxx valide
      (ex: le modèle renvoie "LABEL: Pneumonia" au lieu de "LABEL: L11")
    """
    m = score_metrics(y_true, y_pred)

    print("\n" + "="*70)
    print(name)
    print("="*70)
    print("Fails:", fails)
    print("Accuracy:", round(m["accuracy"], 4))
    print("Macro:    P", round(m["precision_macro"], 4),
          "R", round(m["recall_macro"], 4),
          "F1", round(m["f1_macro"], 4))
    print("Weighted: P", round(m["precision_weighted"], 4),
          "R", round(m["recall_weighted"], 4),
          "F1", round(m["f1_weighted"], 4))

    if show_report:
        print("\n" + classification_report(y_true, y_pred, zero_division=0))

    return m

In [14]:
# ----------------------------
# 11) Prompts P0 / P1 / P2 (SANS CoT)
# - Prompts en français, mais on garde LABEL, DISEASES, etc.
# ----------------------------
SYSTEM_MSG = (
    "Tu es un classificateur de texte strict.\n"
    "Ne donne AUCUNE explication. Ne donne AUCUN conseil médical.\n"
    "Ne répète pas la liste des options.\n"
    "Ta réponse doit se terminer par UNE SEULE ligne au format exact :\n"
    "LABEL: Lxx\n"
)

def build_prompt_P0(txt: str) -> str:
    user = (
        f"Options (codes maladie) :\n{OPTIONS_BLOCK}\n\n"
        f"Texte à classer :\n{txt}\n\n"
        "Réponds avec une seule ligne.\n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG, user)

def build_prompt_P1(txt: str) -> str:
    user = (
        f"DISEASES:\n{OPTIONS_BLOCK}\n\n"
        f"SYMPTOM:\n{txt}\n\n"
        "Réponds avec une seule ligne.\n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG, user)

def build_prompt_P2(txt: str) -> str:
    user = (
        f"DISEASES:\n{OPTIONS_BLOCK}\n\n"
        f"EXAMPLES:\n{fewshot_dynamic(txt, k=DYN_K, per_class_cap=DYN_PER_CLASS_CAP)}"
        f"SYMPTOM:\n{txt}\n\n"
        "Réponds avec une seule ligne.\n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG, user)

In [15]:
# ----------------------------
# 12) Prompts CoT (P3.0 / P3.1 / P3.2 / P3.3)
# ----------------------------
SYSTEM_MSG_COT = (
    "Tu es un classificateur de texte strict.\n"
    "Ne donne AUCUN conseil médical.\n"
    "Tu dois réfléchir brièvement (max 2 puces).\n"
    "Ne répète pas la liste des options.\n"
    "Termine par UNE SEULE ligne finale au format : LABEL: Lxx\n"
)

def build_prompt_P30(txt: str) -> str:
    user = (
        f"SYMPTOM:\n{txt}\n\n"
        "Raisonnement (max 2 puces) :\n- \n- \n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG_COT, user)

def build_prompt_P31(txt: str) -> str:
    user = (
        f"DISEASES:\n{OPTIONS_BLOCK}\n\n"
        f"SYMPTOM:\n{txt}\n\n"
        "Raisonnement (max 2 puces) :\n- \n- \n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG_COT, user)

def build_prompt_P32(txt: str) -> str:
    user = (
        f"DISEASES:\n{OPTIONS_BLOCK}\n\n"
        f"EXAMPLES:\n{FEWSHOT_STATIC_TEXT}"
        f"SYMPTOM:\n{txt}\n\n"
        "Raisonnement (max 2 puces) :\n- \n- \n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG_COT, user)

def build_prompt_P33(txt: str) -> str:
    user = (
        f"DISEASES:\n{OPTIONS_BLOCK}\n\n"
        f"EXAMPLES:\n{fewshot_dynamic(txt, k=DYN_K, per_class_cap=DYN_PER_CLASS_CAP)}"
        f"SYMPTOM:\n{txt}\n\n"
        "Raisonnement (max 2 puces) :\n- \n- \n"
        "LABEL:"
    )
    return chat_template(SYSTEM_MSG_COT, user)

In [16]:
# ----------------------------
# 13) Runner (unifié) — SANS temps (seconds retiré)
# ----------------------------
def run_variant(X_df: pd.DataFrame, builder, max_new_tokens: int = 50, fallback_label_idx: int = 0, desc: str = "Running"):
    """
    Exécute une variante de prompt sur un dataset (ici: TEST),
    parse les codes "Lxx" dans les sorties, et renvoie:
    - y_pred: liste des maladies prédites
    - fails: nb de sorties impossibles à parser (fallback appliqué)
    """
    preds = []
    fails = 0

    for txt in tqdm(X_df["symptom_text"].tolist(), desc=desc):
        prompt = builder(txt)
        gen = generate_text(prompt, max_new_tokens=max_new_tokens)
        c = extract_code_last(gen)

        if (c is None) or (c not in code2idx):
            fails += 1
            preds.append(idx2label[fallback_label_idx])
        else:
            preds.append(idx2label[code2idx[c]])

    return preds, fails

In [18]:
# ----------------------------
# 14) Exécution des variantes + tableau résumé (sans seconds)
# ----------------------------
results = []

def add_result_row(variant_name: str, fails: int, y_true, y_pred):
    m = score_metrics(y_true, y_pred)
    results.append({
        "variant": variant_name,
        "fails": fails,
        **m
    })

# --- P0/P1/P2 (TEST)
for name, builder, tok in [
    ("P0 — Baseline", build_prompt_P0, 40),
    ("P1 — Liste des maladies", build_prompt_P1, 40),
    ("P2 — Liste + few-shot dynamique (TF-IDF)", build_prompt_P2, 50),
]:
    y_pred, fails = run_variant(X_test, builder, max_new_tokens=tok, desc=f"Running {name}")
    print_eval_block(name, y_test, y_pred, fails=fails, show_report=True)
    add_result_row(name, fails, y_test, y_pred)

# --- P3.x (TEST)
for name, builder, tok in [
    ("P3.0 — CoT seul", build_prompt_P30, 70),
    ("P3.1 — CoT + liste maladies", build_prompt_P31, 70),
    ("P3.2 — CoT + few-shot statique", build_prompt_P32, 70),
    ("P3.3 — CoT + few-shot dynamique (TF-IDF)", build_prompt_P33, 70),
]:
    y_pred, fails = run_variant(X_test, builder, max_new_tokens=tok, desc=f"Running {name}")
    print_eval_block(name, y_test, y_pred, fails=fails, show_report=True)
    add_result_row(name, fails, y_test, y_pred)

Running P0 — Baseline: 100%|██████████| 150/150 [1:30:33<00:00, 36.23s/it]



P0 — Baseline
Fails: 0
Accuracy: 0.3
Macro:    P 0.1552 R 0.3 F1 0.1949
Weighted: P 0.1552 R 0.3 F1 0.1949

                       precision    recall  f1-score   support

                 Acne       0.37      1.00      0.54        10
            Arthritis       0.56      1.00      0.71        10
     Bronchial Asthma       0.00      0.00      0.00        10
          Chicken pox       0.17      0.10      0.12        10
          Common Cold       0.00      0.00      0.00        10
               Dengue       0.00      0.00      0.00        10
Dimorphic Hemorrhoids       0.00      0.00      0.00        10
     Fungal infection       0.00      0.00      0.00        10
         Hypertension       0.00      0.00      0.00        10
             Impetigo       0.29      0.20      0.24        10
             Migraine       0.48      1.00      0.65        10
            Pneumonia       0.30      0.90      0.45        10
            Psoriasis       0.00      0.00      0.00        10
        

Running P1 — Liste des maladies:  28%|██▊       | 42/150 [25:38<1:05:56, 36.64s/it]


KeyboardInterrupt: 

In [ ]:
# ----------------------------
# 15) Tableau de synthèse
# ----------------------------
summary = pd.DataFrame(results).sort_values("f1_macro", ascending=False)

print("\n=== SUMMARY (trié par f1_macro) ===")
print(summary[[
    "variant", "fails",
    "accuracy",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted"
]])